# 09. 통계적 유의성 검증

외부 AI 검토 피드백을 반영한 통계 검증 노트북.

**검증 항목**
1. 장르 간 반응률 차이의 통계적 유의성 (Kruskal-Wallis + Pairwise)
2. 장르별 부트스트랩 95% 신뢰구간
3. 할인율 구간별 반응률 단조 증가 (더 강한 신호)
4. 시즌 세일 반응률↑ 유지율↓ 역전 패턴
5. 할인 빈도 통제변수 확인

**결과 활용**: 발표 한계 섹션 및 방어 논리 근거

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu, spearmanr
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings('ignore')
np.random.seed(42)

def root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent]:
        if (c / 'data').exists() and (c / 'figures').exists():
            return c
    return cwd

PROJECT_ROOT = root()
DATA_DIR = PROJECT_ROOT / 'data'
FIGURE_DIR = PROJECT_ROOT / 'figures'

def kfont():
    for fp in [PROJECT_ROOT/'fonts'/'NanumGothic.ttc', PROJECT_ROOT/'fonts'/'NanumGothic.ttf']:
        if fp.exists():
            font_manager.fontManager.addfont(str(fp))
            return font_manager.FontProperties(fname=str(fp)).get_name()
    for fn in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']:
        if fn in {f.name for f in font_manager.fontManager.ttflist}:
            return fn
    return 'DejaVu Sans'

FONT_NAME = kfont()
plt.rcParams.update({'font.family': FONT_NAME, 'axes.unicode_minus': False, 'figure.dpi': 120})

df = pd.read_csv(DATA_DIR / 'analysis_df.csv')
GENRE_ORDER = ['RPG', 'Adventure', 'Strategy/Simulation', 'Casual/Lightweight', 'Action']
DPI = 300

print(f'이벤트 수: {len(df)} / 게임 수: {df["appid"].nunique()}')
print()
print('장르별 이벤트 수:')
print(df.groupby('genre_category')['appid'].count().reindex(GENRE_ORDER).rename('이벤트 수').to_string())

## 검증 1 — 장르 간 반응률 차이의 통계적 유의성

Kruskal-Wallis 검정: p < 0.05이면 장르 간에 통계적으로 의미 있는 차이가 있음.

In [ ]:
# Kruskal-Wallis (이벤트 단위)
groups_ev = [df[df['genre_category']==g]['reaction_rate'].dropna().values for g in GENRE_ORDER]
H_ev, p_ev = kruskal(*groups_ev)

# Kruskal-Wallis (게임 단위)
game_df = df.groupby(['appid','genre_category'])['reaction_rate'].median().reset_index()
groups_gm = [game_df[game_df['genre_category']==g]['reaction_rate'].dropna().values for g in GENRE_ORDER]
H_gm, p_gm = kruskal(*groups_gm)

print('=== Kruskal-Wallis 검정 결과 ===')
print(f'이벤트 단위: H={H_ev:.3f}, p={p_ev:.4f}  ->  {"유의함" if p_ev < 0.05 else "유의하지 않음"}')
print(f'게임 단위:   H={H_gm:.3f}, p={p_gm:.4f}  ->  {"유의함" if p_gm < 0.05 else "유의하지 않음"}')
print()

# Pairwise Mann-Whitney
print('=== Pairwise Mann-Whitney (장르 쌍별) ===')
genre_data = {g: df[df['genre_category']==g]['reaction_rate'].dropna().values for g in GENRE_ORDER}
sig_pairs = []
for g1, g2 in combinations(GENRE_ORDER, 2):
    _, p_mw = mannwhitneyu(genre_data[g1], genre_data[g2], alternative='two-sided')
    marker = '[유의 p<0.05]' if p_mw < 0.05 else ''
    print(f'  {g1:<25} vs {g2:<25} p={p_mw:.4f} {marker}')
    if p_mw < 0.05:
        sig_pairs.append((g1, g2))

print()
if sig_pairs:
    print(f'유의한 쌍: {sig_pairs}')
else:
    print('유의한 쌍 없음 — 모든 장르 쌍에서 p >= 0.05')
    print('-> 장르 순위(Action 1위, RPG 5위 등)는 탐색적 패턴으로 해석 필요')

## 검증 2 — 부트스트랩 95% 신뢰구간

신뢰구간이 겹치면 장르 간 순위를 확정할 수 없음.

In [ ]:
def bootstrap_ci(values, n_boot=2000, ci=95):
    boot = [np.median(np.random.choice(values, len(values), replace=True)) for _ in range(n_boot)]
    lo, hi = np.percentile(boot, [(100-ci)/2, 100-(100-ci)/2])
    return np.median(values), lo, hi

ci_results = {}
print('=== 장르별 부트스트랩 95% 신뢰구간 (이벤트 단위) ===')
for g in GENRE_ORDER:
    vals = genre_data[g]
    med, lo, hi = bootstrap_ci(vals)
    ci_results[g] = (med, lo, hi, len(vals))
    print(f'  {g:<25} n={len(vals):>3}  중앙값={med:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  폭={hi-lo:.3f}')

# 시각화
fig, ax = plt.subplots(figsize=(10, 5))
GENRE_COLORS = {
    'RPG': '#4C72B0', 'Adventure': '#DD8452',
    'Strategy/Simulation': '#55A868', 'Casual/Lightweight': '#C44E52', 'Action': '#8172B2'
}
x = np.arange(len(GENRE_ORDER))
for i, g in enumerate(GENRE_ORDER):
    med, lo, hi, n = ci_results[g]
    ax.bar(i, med, color=GENRE_COLORS[g], alpha=0.75, edgecolor='black')
    ax.errorbar(i, med, yerr=[[med-lo], [hi-med]],
                fmt='none', color='black', capsize=6, linewidth=2)
    ax.text(i, hi + 0.01, f'n={n}', ha='center', va='bottom', fontsize=9)

ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(GENRE_ORDER)
ax.set_ylabel('Engagement 반응률 중앙값')
ax.set_title('장르별 Engagement 반응률 (부트스트랩 95% 신뢰구간)')
ax.text(0.02, 0.97, f'K-W p={p_ev:.3f} — 장르 간 차이 통계적 미유의\n오차막대: 부트스트랩 95% CI (반복 2,000회)',
        transform=ax.transAxes, va='top', fontsize=8, color='gray',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart12_ci.png', dpi=DPI, bbox_inches='tight')
plt.close()
print()
print('저장 완료 chart12_ci.png')

## 검증 3 — 할인율 구간별 반응률 (더 강한 신호)

장르 차이(p=0.52)보다 할인 깊이 효과(p<0.05)가 더 강한 신호.

In [ ]:
# 할인율 구간
bins   = [0, 25, 50, 75, 100]
labels = ['~25%', '25~50%', '50~75%', '75%+']
df['discount_bin'] = pd.cut(df['discount_pct'], bins=bins, labels=labels)

disc_summary = (
    df.groupby('discount_bin', observed=True)['reaction_rate']
    .agg(['median', 'count'])
    .rename(columns={'median': '반응률 중앙값', 'count': 'n'})
)

rho, p_rho = spearmanr(df['discount_pct'], df['reaction_rate'])

print('=== 할인율 구간별 Engagement 반응률 ===')
print(disc_summary.round(3).to_string())
print()
print(f'Spearman rho = {rho:.3f}, p = {p_rho:.4f}')
print(f'-> {"유의함 — 할인율 높을수록 반응률 높아지는 경향 통계적으로 확인" if p_rho < 0.05 else "유의하지 않음"}')

# 시각화
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(range(len(labels)), disc_summary['반응률 중앙값'],
              color=['#BDD7EE','#9DC3E6','#5B9BD5','#2E75B6'], edgecolor='black', alpha=0.85)
for bar, (idx, row) in zip(bars, disc_summary.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{row["반응률 중앙값"]:+.3f}\n(n={int(row["n"])})', ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_xlabel('할인율 구간')
ax.set_ylabel('Engagement 반응률 중앙값')
ax.set_title('할인율이 높을수록 반응률도 높아진다')
ax.text(0.98, 0.05, f'Spearman rho={rho:.3f}, p={p_rho:.4f}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart13_discount_depth.png', dpi=DPI, bbox_inches='tight')
plt.close()
print()
print('저장 완료 chart13_discount_depth.png')

## 검증 4 — 시즌 세일 역전 패턴

시즌 세일은 반응률은 높지만 유지율이 오히려 낮아지는 역전 패턴.

In [ ]:
season = df.groupby('is_seasonal_sale')[['reaction_rate','sustained_rate']].median().round(3)
season.index = season.index.map({False: '비시즌', True: '시즌 세일'})
season.columns = ['Engagement 반응률', 'Engagement 유지율']
counts = df.groupby('is_seasonal_sale')['appid'].count()
counts.index = counts.index.map({False: '비시즌', True: '시즌 세일'})
season['이벤트 수'] = counts

print('=== 시즌 세일 vs 비시즌 ===')
print(season.to_string())
print()

rr_diff = season.loc['시즌 세일','Engagement 반응률'] - season.loc['비시즌','Engagement 반응률']
sr_diff = season.loc['시즌 세일','Engagement 유지율'] - season.loc['비시즌','Engagement 유지율']
print(f'반응률 차이: {rr_diff:+.3f}  (시즌이 {"높음" if rr_diff>0 else "낮음"})')
print(f'유지율 차이: {sr_diff:+.3f}  (시즌이 {"높음" if sr_diff>0 else "낮음"}) <- 역전 패턴')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, col, title in zip(axes,
    ['Engagement 반응률', 'Engagement 유지율'],
    ['Engagement 반응률 (시즌 vs 비시즌)', 'Engagement 유지율 (시즌 vs 비시즌)']):
    vals = season[col]
    bars = ax.bar(vals.index, vals, color=['#5B9BD5','#ED7D31'], edgecolor='black', alpha=0.85)
    for bar, idx in zip(bars, vals.index):
        n = int(season.loc[idx, '이벤트 수'])
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{vals[idx]:+.3f}\n(n={n})', ha='center', va='bottom', fontsize=10)
    ax.axhline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_title(title)
    ax.set_ylabel('중앙값')
    ax.grid(axis='y', alpha=0.3)

axes[1].text(0.5, 0.05, '시즌 세일: 반응률 높지만 유지율 낮아지는 역전 패턴',
             transform=axes[1].transAxes, ha='center', va='bottom', fontsize=8, color='dimgray')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart14_seasonal.png', dpi=DPI, bbox_inches='tight')
plt.close()
print()
print('저장 완료 chart14_seasonal.png')

## 검증 5 — 할인 빈도 통제변수 확인

저빈도 vs 고빈도 차이가 장르·할인율·시즌 구성 차이 때문인지 확인.

In [ ]:
freq = df.groupby('appid').agg(
    n_events=('appid','count'),
    reaction_med=('reaction_rate','median'),
    genre=('genre_category','first'),
    avg_discount=('discount_pct','mean'),
    seasonal_ratio=('is_seasonal_sale','mean'),
    before_avg=('before_daily_avg','mean'),
).reset_index()

cut = int(freq['n_events'].median())
freq['group'] = freq['n_events'].apply(lambda x: f'고빈도({cut+1}건+)' if x > cut else f'저빈도({cut}건 이하)')

ctrl = freq.groupby('group')[['reaction_med','avg_discount','seasonal_ratio','before_avg']].mean().round(3)
ctrl.columns = ['반응률 중앙값','평균 할인율(%)','시즌 세일 비율','할인 전 일평균 리뷰']
ctrl['게임 수'] = freq.groupby('group')['appid'].count()

print('=== 할인 빈도별 통제변수 비교 ===')
print(ctrl.to_string())
print()

rho_f, p_f = spearmanr(freq['n_events'], freq['reaction_med'])
print(f'빈도-반응률 Spearman rho={rho_f:.3f}, p={p_f:.4f}')
print()
print('해석: 통제변수(할인율·시즌비율·인기도) 차이가 함께 보이면')
print('     빈도 효과를 희소성 하나로 단정하기 어려움 — 탐색적 패턴으로 제시')

## 결론 요약 — 발표 방어 논리

## 검증 6 — RPG 필터 역설: 표본 구성 변화 없음 확인

외부 검토 피드백: "필터를 강하게 걸면 이벤트 구성 자체가 바뀌는 거 아니냐?"

→ 세 기준(전체/2시간+/10시간+) 모두에서 살아남은 **공통 이벤트만** 대상으로 재비교.

In [ ]:
rv = pd.read_csv(DATA_DIR / 'review_individual.csv')
disc_all = pd.read_csv(DATA_DIR / 'discount_history.csv')
disc_all['discount_start'] = pd.to_datetime(disc_all['discount_start'])
disc_all['discount_end']   = pd.to_datetime(disc_all['discount_end'])
rv['date'] = pd.to_datetime(rv['timestamp_created'], unit='s', utc=True).dt.tz_localize(None).dt.normalize()

PRE_D, POST_D = 30, 14

def window_avg_sr(s, start, end):
    sub = s[(s.index >= start) & (s.index < end)]
    return sub.sum() / (end - start).days if len(sub) else np.nan

def compute_sustained(rv_f, disc_sub):
    daily = rv_f.groupby(['app_id','date'])['review_id'].count().reset_index()
    daily.columns = ['appid','date','cnt']
    daily['date'] = pd.to_datetime(daily['date'])
    smap = {a: g.set_index('date')['cnt'] for a, g in daily.groupby('appid')}
    rows = []
    for _, ev in disc_sub.iterrows():
        appid = int(ev['appid'])
        if appid not in smap: continue
        s = smap[appid]
        st, en = ev['discount_start'], ev['discount_end']
        if (st - s.index.min()).days < 14: continue
        if (s.index.max() - en).days < 7: continue
        b = window_avg_sr(s, st - pd.Timedelta(days=PRE_D), st)
        a = window_avg_sr(s, en, en + pd.Timedelta(days=POST_D))
        if pd.isna(b) or b == 0: continue
        rows.append({'appid': appid, 'key': f'{appid}_{st}',
                     'sr': (a - b) / b if not pd.isna(a) else np.nan})
    return pd.DataFrame(rows).dropna(subset=['sr'])

rpg_disc = disc_all[disc_all['genre_category'] == 'RPG']
filter_sets = {
    '전체':   rv,
    '2시간+': rv[rv['playtime_at_review_min'] >= 120],
    '10시간+': rv[rv['playtime_at_review_min'] >= 600],
}

results_sr = {}
for label, rv_f in filter_sets.items():
    res = compute_sustained(rv_f, rpg_disc)
    results_sr[label] = res.set_index('key')['sr']

# 공통 이벤트
common_keys = set(results_sr['전체'].index)
for label in ['2시간+', '10시간+']:
    common_keys &= set(results_sr[label].index)

print(f'RPG 유효 이벤트 수: 전체={len(results_sr["전체"])} / 2시간+={len(results_sr["2시간+"])} / 10시간+={len(results_sr["10시간+"])}')
print(f'세 기준 공통 이벤트: {len(common_keys)}개')
print()
print('=== RPG Engagement 유지율 비교 ===')
print(f'{"기준":<10}  {"전체 이벤트":>14}  {"공통 이벤트만":>14}')
print('-' * 44)
for label in filter_sets:
    med_all    = results_sr[label].median()
    med_common = results_sr[label].loc[list(common_keys)].median()
    print(f'{label:<10}  {med_all:>+14.4f}  {med_common:>+14.4f}')

print()
print('결론: 공통 이벤트 기준에서도 유지율이 상승하면')
print('      "표본 구성이 바뀐 것"이 아니라 "진짜 신호"임을 확인')

In [ ]:
print('=' * 60)
print('통계 검증 결과 요약')
print('=' * 60)
print()
print('[1] 장르 간 반응률 차이')
print(f'    K-W p={p_ev:.4f} — 통계적으로 유의하지 않음')
print('    -> 장르 순위는 탐색적 패턴으로 제시. 확정 결론 아님.')
print()
print('[2] 더 강한 신호 — 할인율')
print(f'    Spearman rho={rho:.3f}, p={p_rho:.4f} — 통계적으로 유의')
print('    -> 할인율 높을수록 반응률 높아지는 단조 증가 확인')
print()
print('[3] 시즌 세일 역전 패턴')
print(f'    반응률: 시즌 {season.loc["시즌 세일","Engagement 반응률"]:+.3f} vs 비시즌 {season.loc["비시즌","Engagement 반응률"]:+.3f}')
print(f'    유지율: 시즌 {season.loc["시즌 세일","Engagement 유지율"]:+.3f} vs 비시즌 {season.loc["비시즌","Engagement 유지율"]:+.3f}')
print('    -> 시즌 세일은 단기 폭증·장기 효과 미약')
print()
print('발표 권장 메시지:')
print('  "장르 차이는 탐색적 패턴이며, 더 강한 신호는')
print('   할인율(p<0.05)과 시즌성(유지율 역전)입니다."')
print('=' * 60)